In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
stop_words = set(stopwords.words("english"))
from nltk.stem import PorterStemmer
from nltk.stem.wordnet import WordNetLemmatizer
lemma = WordNetLemmatizer()
ps = PorterStemmer()
import re


from scipy.spatial import distance
from scipy.spatial import minkowski_distance
from scipy.spatial.distance import cosine




[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/sophie/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
#Electronics Dataset:

import numpy as np
import pandas as pd
fileElectr='amazon_reviews_us_Electronics_v1_00.tsv'
df=pd.read_csv(fileElectr, sep="\t", header=0, on_bad_lines='skip')
df=df.dropna(subset=['review_headline', 'review_body', 'star_rating'])

In [4]:
df.star_rating.value_counts()

5    1779304
4     536398
1     357800
3     238378
2     179025
Name: star_rating, dtype: int64

In [13]:
from transformers import pipeline


    
    
def emotion_roberta(df_sample, column_name):
    classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", return_all_scores=True)

    for i in range (0, len(df_sample[column_name])):

        #processed text
        text=df_sample[column_name][i]  

        if len(text) > 512:
            text = text[:512]

        prediction = classifier(text )

        for j in range(0,7):  #loop over the  emotions:
            df_sample.loc[i, ("roberta_"+column_name+prediction[0][j]['label'])]=prediction[0][j]['score'] #BP=body processed


        if i%1000==0:
           print ("\n emotion_roberta:   We are at i=", str(i))    
            
            
    return df_sample




In [6]:
def text_process2(reviews, column_name):  #input is the dataframe
    for i  in range(0, reviews[column_name].count()):
       review_body=reviews.loc[i, (column_name)]  #tokens= word_tokenize(df_sample.loc[1, ('review_body')])
       review_body=re.sub('<br\s?\/>|<br>', " ", review_body)  #remove the br
       tokens= word_tokenize(review_body)
       tokens = [w.lower()  for w in tokens ]
       #tokens = [w for w in tokens if not w in stop_words]
       tokens = [w for w in tokens if w.isalpha()] #remove non alphabetic items like like 5 or ;
       tokens = [lemma.lemmatize(w) for w in tokens]
       # tokens = [ps.stem(w) for w in tokens]
       column_name_out=column_name+"_processed"
       reviews.loc[i, (column_name_out)]=' '.join(tokens)
       
       if i%10000==0:
           print ("\n text_process:   We are at i=", str(i))
       
    return reviews





In [14]:
n_samples=10000

N_rewiews=df.loc[df['star_rating'] == 1].sample(n_samples, replace=False, random_state=1900)
N_rewiew2=df.loc[df['star_rating'] == 5].sample(n_samples, replace=False, random_state=1900)


samplesize=n_samples*2

N_rewiews=N_rewiews.append(N_rewiew2)



/var/folders/ds/k83592y50w34wqwchx8qldph0000gn/T/ipykernel_5253/3763742897.py:9: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  N_rewiews=N_rewiews.append(N_rewiew2)


In [15]:
N_rewiews=N_rewiews.reset_index()

In [16]:
#N_rewiews=text_process2(N_rewiews,'review_body')
N_rewiews=text_process2(N_rewiews,'review_headline')







 text_process:   We are at i= 0

 text_process:   We are at i= 10000


In [17]:
emotion_roberta(N_rewiews, 'review_body')
#emotion_distilbert(N_rewiews, 'review_body')
#
#emotion_roberta(N_rewiews, 'review_body_processed')
#
#emotion_distilbert(N_rewiews, 'review_body_processed')




 emotion_roberta:   We are at i= 0

 emotion_roberta:   We are at i= 1000

 emotion_roberta:   We are at i= 2000

 emotion_roberta:   We are at i= 3000

 emotion_roberta:   We are at i= 4000

 emotion_roberta:   We are at i= 5000

 emotion_roberta:   We are at i= 6000

 emotion_roberta:   We are at i= 7000

 emotion_roberta:   We are at i= 8000

 emotion_roberta:   We are at i= 9000

 emotion_roberta:   We are at i= 10000

 emotion_roberta:   We are at i= 11000

 emotion_roberta:   We are at i= 12000

 emotion_roberta:   We are at i= 13000

 emotion_roberta:   We are at i= 14000

 emotion_roberta:   We are at i= 15000

 emotion_roberta:   We are at i= 16000

 emotion_roberta:   We are at i= 17000

 emotion_roberta:   We are at i= 18000

 emotion_roberta:   We are at i= 19000


,index,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,...,review_body,review_date,review_headline_processed,roberta_review_bodyanger,roberta_review_bodydisgust,roberta_review_bodyfear,roberta_review_bodyjoy,roberta_review_bodyneutral,roberta_review_bodysadness,roberta_review_bodysurprise
0,1677873,US,18585476,R31LJGNJRWGRS,B003ARSOWQ,864558418,Timex T715BW3 Dual Alarm Clock Radio (Black),Electronics,1,0,...,"I have had the product for just under a month,...",2013-12-14,doe it keep time not really,0.148382,0.261074,0.011709,0.001612,0.522428,0.038445,0.016350
1,1856468,US,51075252,R35YF0DWJE87A4,B0044WS7KK,471031907,"Aerial7 Perisher - Black - black, one size",Electronics,1,0,...,The speaker quality reminds you of a 1960's tr...,2013-08-12,very poor sound,0.002942,0.005885,0.005614,0.006947,0.032595,0.002518,0.943499
2,600712,US,15700552,R2PQJHUI1SF24E,B00INO6JX2,703104763,Samsung SSG-5150GB 3D Active Glasses,Electronics,1,1,...,Did not work,2015-02-24,wrong item,0.015367,0.015098,0.006451,0.001496,0.121806,0.820625,0.019158
3,2871105,US,26397372,R1B30LZ8X5V07Q,B00008VSK5,50332,Acoustic Research MS805 Adaptatip Flex Pin,Electronics,1,4,...,This flex pin set requires an additional part:...,2008-12-22,flex pin set incomplete not worth it,0.260976,0.092722,0.015083,0.002443,0.487012,0.094730,0.047033
4,1381712,US,20903669,R2B9JFV8C5AHX,B007N16IYG,623454097,Panasonic Deep Base Ergo-Fit Inner Ear Earbud ...,Electronics,1,1,...,These are not decent. This earphone set is spl...,2014-05-16,these are not what you think much regret,0.052197,0.902236,0.003806,0.000668,0.018588,0.015990,0.006515
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,2273697,US,30170180,R1JG9WFRJH44NB,B003XO57VM,846711659,Philips Lighting Sony KDF-60XS955 KDF60XS955 L...,Electronics,5,2,...,"Came promptly, easy to install (I'm not handy)...",2012-11-11,oem quality easy install,0.017919,0.004830,0.005972,0.095326,0.377220,0.005577,0.493156
19996,1200249,US,44475306,R2MMLE26RF3WC2,B00358VSI2,599267792,Coby Digital Active Noise-Canceling Stereo Hea...,Electronics,5,0,...,I wanted noise cancelling headphones to use wh...,2014-08-12,nice headphone,0.007445,0.013562,0.007493,0.078569,0.851484,0.025257,0.016191
19997,852624,US,46755668,R1X86QEU74V7LK,B00BN0N0N0,21856130,Sony MDRAS200 Active Sports Headphones,Electronics,5,0,...,"Sound is excellent, they stay put running, wal...",2014-12-20,five star,0.004954,0.013535,0.002225,0.279442,0.657715,0.031152,0.010976
19998,757939,US,40385685,R2JLSZ9EJHV4XS,B0077QMHVU,311150041,WILSON ANTENNAS Noise Canceling CB Microphone ...,Electronics,5,0,...,Excellent ممتازه,2015-01-12,five star,0.019810,0.023431,0.007185,0.047440,0.887383,0.006005,0.008747


In [18]:
emotion_roberta(N_rewiews, 'review_headline')
#emotion_distilbert(N_rewiews, 'review_headline')


emotion_roberta(N_rewiews, 'review_headline_processed')
#emotion_distilbert(N_rewiews, 'review_headline_processed')





 emotion_roberta:   We are at i= 0

 emotion_roberta:   We are at i= 1000

 emotion_roberta:   We are at i= 2000

 emotion_roberta:   We are at i= 3000

 emotion_roberta:   We are at i= 4000

 emotion_roberta:   We are at i= 5000

 emotion_roberta:   We are at i= 6000

 emotion_roberta:   We are at i= 7000

 emotion_roberta:   We are at i= 8000

 emotion_roberta:   We are at i= 9000

 emotion_roberta:   We are at i= 10000

 emotion_roberta:   We are at i= 11000

 emotion_roberta:   We are at i= 12000

 emotion_roberta:   We are at i= 13000

 emotion_roberta:   We are at i= 14000

 emotion_roberta:   We are at i= 15000

 emotion_roberta:   We are at i= 16000

 emotion_roberta:   We are at i= 17000

 emotion_roberta:   We are at i= 18000

 emotion_roberta:   We are at i= 19000

 emotion_roberta:   We are at i= 0

 emotion_roberta:   We are at i= 1000

 emotion_roberta:   We are at i= 2000

 emotion_roberta:   We are at i= 3000

 emotion_roberta:   We are at i= 4000

 emotion_roberta:   

,index,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,...,roberta_review_headlineneutral,roberta_review_headlinesadness,roberta_review_headlinesurprise,roberta_review_headline_processedanger,roberta_review_headline_processeddisgust,roberta_review_headline_processedfear,roberta_review_headline_processedjoy,roberta_review_headline_processedneutral,roberta_review_headline_processedsadness,roberta_review_headline_processedsurprise
0,1677873,US,18585476,R31LJGNJRWGRS,B003ARSOWQ,864558418,Timex T715BW3 Dual Alarm Clock Radio (Black),Electronics,1,0,...,0.683399,0.037004,0.196546,0.010669,0.017760,0.007133,0.007847,0.839019,0.079229,0.038343
1,1856468,US,51075252,R35YF0DWJE87A4,B0044WS7KK,471031907,"Aerial7 Perisher - Black - black, one size",Electronics,1,0,...,0.407534,0.376775,0.032329,0.022509,0.095116,0.019774,0.002500,0.122202,0.713169,0.024730
2,600712,US,15700552,R2PQJHUI1SF24E,B00INO6JX2,703104763,Samsung SSG-5150GB 3D Active Glasses,Electronics,1,1,...,0.031080,0.012691,0.002935,0.042930,0.037332,0.011042,0.004079,0.698276,0.124367,0.081975
3,2871105,US,26397372,R1B30LZ8X5V07Q,B00008VSK5,50332,Acoustic Research MS805 Adaptatip Flex Pin,Electronics,1,4,...,0.309574,0.564732,0.034902,0.031687,0.021541,0.011392,0.006795,0.329728,0.535445,0.063412
4,1381712,US,20903669,R2B9JFV8C5AHX,B007N16IYG,623454097,Panasonic Deep Base Ergo-Fit Inner Ear Earbud ...,Electronics,1,1,...,0.426625,0.491464,0.024036,0.013662,0.043480,0.015211,0.003589,0.608421,0.278068,0.037570
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,2273697,US,30170180,R1JG9WFRJH44NB,B003XO57VM,846711659,Philips Lighting Sony KDF-60XS955 KDF60XS955 L...,Electronics,5,2,...,0.953282,0.003900,0.017110,0.005241,0.002233,0.001612,0.214094,0.685456,0.016548,0.074817
19996,1200249,US,44475306,R2MMLE26RF3WC2,B00358VSI2,599267792,Coby Digital Active Noise-Canceling Stereo Hea...,Electronics,5,0,...,0.800938,0.017631,0.028434,0.004495,0.024509,0.002833,0.151292,0.750572,0.025584,0.040714
19997,852624,US,46755668,R1X86QEU74V7LK,B00BN0N0N0,21856130,Sony MDRAS200 Active Sports Headphones,Electronics,5,0,...,0.874900,0.009004,0.035351,0.017302,0.005385,0.002486,0.472787,0.424965,0.012054,0.065021
19998,757939,US,40385685,R2JLSZ9EJHV4XS,B0077QMHVU,311150041,WILSON ANTENNAS Noise Canceling CB Microphone ...,Electronics,5,0,...,0.874900,0.009004,0.035351,0.017302,0.005385,0.002486,0.472787,0.424965,0.012054,0.065021


In [ ]:
#NOT USED 
#Roberta_Body_Processed=[
#'roberta_review_body_processedanger',
#'roberta_review_body_processeddisgust',
#'roberta_review_body_processedfear',
#'roberta_review_body_processedjoy',
#'roberta_review_body_processedneutral',
#'roberta_review_body_processedsadness',
#'roberta_review_body_processedsurprise' ] 

In [19]:
#DETAILED REVIEW for Roberta

Roberta_Body=['roberta_review_bodyanger',
'roberta_review_bodydisgust',
'roberta_review_bodyfear',
'roberta_review_bodyjoy',
'roberta_review_bodyneutral',
'roberta_review_bodysadness',
'roberta_review_bodysurprise']



Roberta_Head=[
'roberta_review_headlineanger', 
'roberta_review_headlinedisgust',
'roberta_review_headlinefear', 
'roberta_review_headlinejoy',
'roberta_review_headlineneutral', 
'roberta_review_headlinesadness',
'roberta_review_headlinesurprise']


Roberta_Head_Processed=[
'roberta_review_headline_processedanger', 
'roberta_review_headline_processeddisgust',
'roberta_review_headline_processedfear', 
'roberta_review_headline_processedjoy',
'roberta_review_headline_processedneutral', 
'roberta_review_headline_processedsadness',
'roberta_review_headline_processedsurprise'] 





In [27]:

from sklearn.model_selection import train_test_split

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix





def run_SVC(df_sample_all_test,N_rewiews):
    target_names = ['0 = rating of 1',  '1 = rating of 5'] # 0 = negative, 4 = positive


    X_train, X_test, y_train, y_test = train_test_split(df_sample_all_test, N_rewiews['star_rating'],  random_state=56)

    clf = make_pipeline(StandardScaler(), SVC( C=1000, gamma= 0.001, kernel= 'rbf'))
    clf.fit(X_train, y_train)
    y_pred=clf.predict(X_test)

    clf.score(X_test, y_test)
    y_test.value_counts()
    print(clf.score(X_test, y_test))

    print(classification_report(y_test, y_pred, target_names=target_names, digits=6))

    print(confusion_matrix(y_test, y_pred))




In [21]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.datasets import make_classification


def run_AdaBoost(df_sample_all_test,N_rewiews):

    X_train, X_test, y_train, y_test = train_test_split(df_sample_all_test, N_rewiews['star_rating'],  random_state=56)

    clf = AdaBoostClassifier(n_estimators=50, random_state=156)
    
    clf.fit(X_train, y_train)
    y_pred=clf.predict(X_test)

    print(clf.score(X_test, y_test))

#    clf.score(X_test, y_test)
    target_names = ['0 = rating of 1',  '1 = rating of 5'] # 0 = negative, 4 = positive


    print(classification_report(y_test, y_pred, target_names=target_names, digits=6))

    print(confusion_matrix(y_test, y_pred))







In [28]:
df_sample_all_test=N_rewiews[ Roberta_Body+Roberta_Head+Roberta_Head_Processed ] 

run_SVC(df_sample_all_test,N_rewiews)


0.931
                 precision    recall  f1-score   support

0 = rating of 1   0.937046  0.924731  0.930848      2511
1 = rating of 5   0.925059  0.937324  0.931151      2489

       accuracy                       0.931000      5000
      macro avg   0.931053  0.931028  0.931000      5000
   weighted avg   0.931079  0.931000  0.930999      5000

[[2322  189]
 [ 156 2333]]


In [29]:
run_AdaBoost(df_sample_all_test,N_rewiews)



0.939
                 precision    recall  f1-score   support

0 = rating of 1   0.944042  0.933891  0.938939      2511
1 = rating of 5   0.934022  0.944154  0.939061      2489

       accuracy                       0.939000      5000
      macro avg   0.939032  0.939023  0.939000      5000
   weighted avg   0.939054  0.939000  0.939000      5000

[[2345  166]
 [ 139 2350]]


In [30]:
#adding embeddings 

In [31]:
def df2emd(word2vec_model, N_rewiews, column_name):
    word2vec_model_embeddings = WordVecVectorizer(word2vec_model)

    word2vec_model_embeddings_ave_one_review_list=[]
    embed_only=pd.DataFrame()
    
    for i in range(0,len(N_rewiews[column_name]) ) : 
    #for i in range(0,len(N_rewiews['review_body_process']) ) : 
        #list_words=[N_rewiews['review_body_process'][i]]
        list_words=[N_rewiews[column_name][i]]

        list_words=check_against_word2vec_model(list_words, word2vec_model)
        word2vec_embeddings_one_review=word2vec_model_embeddings.transform(list_words)
        word2vec_model_embeddings_ave_one_review_list.append(word2vec_embeddings_one_review)

    embed_only=pd.DataFrame(np.concatenate(word2vec_model_embeddings_ave_one_review_list))
    
    return embed_only

In [32]:

class WordVecVectorizer(object):
    def __init__(self, word2vec_model):
        self.word2vec_model = word2vec_model
        self.dim = 300
    def transform(self, X):
        return np.array([
            np.mean([self.word2vec_model[w] for w in texts.split() if w in self.word2vec_model]
                    or [np.zeros(self.dim)], axis=0)
            for texts in X
        ])




def check_against_word2vec_model(list_topics, word2vec_model):
    for i  in range(0, len(list_topics) ):
       tokens= word_tokenize(list_topics[i])
       tokens = [w for w in tokens if w in word2vec_model.key_to_index ]
       list_topics[i]=' '.join(tokens)
       return list_topics







In [33]:
import gensim


file_embeddings_fast='crawl-300d-2M.vec'


word2vec_model_fast = gensim.models.KeyedVectors.load_word2vec_format(file_embeddings_fast) 
print(word2vec_model_fast.vector_size)





300


In [34]:
embed_only_fast_B=df2emd(word2vec_model_fast, N_rewiews, "review_body")
embed_only_fast_HP=df2emd(word2vec_model_fast, N_rewiews, "review_headline")


embed_combined=embed_only_fast_B.join(embed_only_fast_HP, lsuffix='_caller', rsuffix='_other')




In [35]:
embed_combined.shape

(20000, 600)

In [36]:
run_SVC(embed_combined,N_rewiews)



0.9416
                 precision    recall  f1-score   support

0 = rating of 1   0.951935  0.930705  0.941200      2511
1 = rating of 5   0.931631  0.952591  0.941994      2489

       accuracy                       0.941600      5000
      macro avg   0.941783  0.941648  0.941597      5000
   weighted avg   0.941827  0.941600  0.941596      5000

[[2337  174]
 [ 118 2371]]


In [37]:
#combine the embeddings with the emotions

embed_emptions_combined=embed_combined.join(df_sample_all_test, lsuffix='_caller', rsuffix='_other')



In [38]:
df_sample_all_test.shape

(20000, 21)

In [39]:
embed_emptions_combined.shape

(20000, 621)

In [40]:
run_SVC(embed_emptions_combined,N_rewiews)

0.9526
                 precision    recall  f1-score   support

0 = rating of 1   0.961820  0.943051  0.952343      2511
1 = rating of 5   0.943656  0.962234  0.952855      2489

       accuracy                       0.952600      5000
      macro avg   0.952738  0.952642  0.952599      5000
   weighted avg   0.952778  0.952600  0.952597      5000

[[2368  143]
 [  94 2395]]


In [41]:
run_AdaBoost(embed_emptions_combined,N_rewiews)

0.9498
                 precision    recall  f1-score   support

0 = rating of 1   0.951639  0.948228  0.949930      2511
1 = rating of 5   0.947958  0.951386  0.949669      2489

       accuracy                       0.949800      5000
      macro avg   0.949799  0.949807  0.949800      5000
   weighted avg   0.949807  0.949800  0.949800      5000

[[2381  130]
 [ 121 2368]]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(embed_emptions_combined, N_rewiews['star_rating'],  random_state=56)



#https://scikit-learn.org/stable/auto_examples/model_selection/plot_grid_search_digits.html
#Grid search tuning

from sklearn.model_selection import GridSearchCV

tuned_parameters =  [ {"kernel": ["rbf"], "gamma": [1e-3, 1e-4], "C": [1, 10, 100, 1000]}, {"kernel": ["linear"], "C": [1, 10, 100, 1000]},  ]  
    
scores = ["precision", "recall"]

for score in scores:
    print("# Tuning hyper-parameters for %s" % score)
    print()

    clf = GridSearchCV(SVC(), tuned_parameters, scoring="%s_macro" % score)
    clf.fit(X_train, y_train)

    print("Best parameters set found on Train set:")
    print()
    print(clf.best_params_)
    print()
    print("Grid scores on Train set:")
    print()
    means = clf.cv_results_["mean_test_score"]
    stds = clf.cv_results_["std_test_score"]
    for mean, std, params in zip(means, stds, clf.cv_results_["params"]):
        print("%0.3f (+/-%0.03f) for %r" % (mean, std * 2, params))
    print()

    print("Report for y_true and y_pred:")
    print()
    y_true, y_pred = y_test, clf.predict(X_test)
    print(classification_report(y_true, y_pred))
    print()

# Tuning hyper-parameters for precision



In [ ]:
#https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html#sklearn.tree.DecisionTreeClassifier
#REF :https://stackoverflow.com/questions/32210569/using-gridsearchcv-with-adaboost-and-decisiontreeclassifier

from sklearn.tree import DecisionTreeClassifier
#from sklearn.grid_search import GridSearchCV

Tree = AdaBoostClassifier(base_estimator=DecisionTreeClassifier(), random_state = 11)

parameters = {'base_estimator__max_depth':[i for i in range(2,20,1)],
              'base_estimator__min_samples_leaf':[1, 5,10, 15],
              'n_estimators':[10,50,100],
              'learning_rate':[1e-3, 1e-4, 0.01,0.1]}

clf = GridSearchCV(Tree, parameters,verbose=3,scoring='f1',n_jobs=-1)
clf.fit(X_train,y_train)

In [ ]:
#https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html
#REF :https://stackoverflow.com/questions/32210569/using-gridsearchcv-with-adaboost-and-decisiontreeclassifier

#from sklearn.ensemble import AdaBoostClassifier
#from sklearn.grid_search import GridSearchCV

param_grid = {"base_estimator__criterion" : ["gini", "entropy"],
              "base_estimator__splitter" :   ["best", "random"],
              "n_estimators": [1, 2, 5, 10, 30], 
              "learning_rate": [1e-3, 1e-4, 0.01,0.1]             }

Ada = AdaBoostClassifier(base_estimator = DTC, random_state=155)

# run grid search
grid_search_ABC = GridSearchCV(ABC, param_grid=param_grid, scoring = 'roc_auc')